# Build Waves setup helper

Validate, deploy and set up one genie-build-waves **instance** (one project) from a notebook, on serverless or classic compute. Everything project-specific comes from the project's **vars file** (the JSON passed to `scripts/deploy_instance.py --vars`), so the same notebook works for every project.

| Step | Cell | What it does | Needs |
|---|---|---|---|
| 1 | Parameters | Widgets: vars file, target, action | |
| 2 | Load configuration | Reads the vars file, checks required keys and the monitor variables | Read access to the vars file |
| 3 | Validate the manifest | `waves_manifest.validate_manifest()` (the same check setup runs), plan and context files exist, manifest `branch` matches `build_branch` | Read access to `build_root` |
| 4 | Deploy command | Prints the `deploy_instance.py` commands. **Run them in the web terminal**: the Databricks CLI does not run on serverless | Web terminal |
| 5 | Find the instance's jobs | Reads the instance's remote bundle state, checks the jobs belong to this instance | A deploy (step 4) |
| 6 | Run setup | `action = run_setup` only: runs the setup job and waits for it | `CAN_MANAGE_RUN` on the setup job |
| 7 | Results | Job IDs, published skills, automations, monitor grants, smoke result | |

Run with `action = check` first (steps 1-5, read-only), then deploy in the terminal, then set `action = run_setup` and **Run all**.

This notebook never deploys, never starts a build run, and never changes permissions. With an external monitor at level 4, the monitor starts the first build run; otherwise start it from the Jobs UI.

In [0]:
%pip install --upgrade databricks-sdk --quiet
dbutils.library.restartPython()

In [0]:
# Project-specific values live in the vars file, not here.
dbutils.widgets.text("vars_file", "", "1. Vars file (JSON, workspace path)")
dbutils.widgets.dropdown("target", "dev", ["dev", "prod"], "2. Bundle target")
dbutils.widgets.dropdown("action", "check", ["check", "run_setup"], "3. Action")
dbutils.widgets.dropdown("run_smoke", "false", ["false", "true"], "4. Run the smoke job in setup")
dbutils.widgets.text("wait_minutes", "60", "5. Max wait for setup (minutes)")
# Optional overrides; leave blank for the defaults.
dbutils.widgets.text("bundle_dir", "", "6. genie-build-waves folder (blank = this notebook's)")
dbutils.widgets.text("root_path", "", "7. Instance root_path (blank = dev default)")

In [0]:
import json
import os
import shlex
import sys

from databricks.sdk import WorkspaceClient

BUNDLE_NAME = "genie-build-waves"  # bundle.name in databricks.yml
REQUIRED_VARS = ["project_name", "instance_suffix", "build_root", "build_branch", "manifest_path",
                 "skills_catalog", "skills_schema"]

w = WorkspaceClient()  # notebook authentication; no tokens handled here
me = w.current_user.me().user_name


def ws_path(p: str) -> str:
    """Workspace path as a local file path (/Workspace/...)."""
    p = (p or "").strip().rstrip("/")
    return p if p.startswith("/Workspace/") or not p.startswith("/") else "/Workspace" + p


def api_path(p: str) -> str:
    """Workspace path for the Workspace API (no /Workspace prefix)."""
    p = ws_path(p)
    return p[len("/Workspace"):] if p.startswith("/Workspace/") else p


vars_file = ws_path(dbutils.widgets.get("vars_file"))
target = dbutils.widgets.get("target")
action = dbutils.widgets.get("action")
run_smoke = dbutils.widgets.get("run_smoke")
wait_minutes = int(dbutils.widgets.get("wait_minutes") or "60")

if not vars_file:
    raise ValueError("Set the vars_file widget to the project's vars JSON (the file passed to deploy_instance.py --vars).")
if not os.path.exists(vars_file):
    raise FileNotFoundError(f"vars file not found: {vars_file}")

# genie-build-waves folder: the widget, else the parent of this notebook's folder (scripts/).
bundle_dir = dbutils.widgets.get("bundle_dir").strip()
if not bundle_dir:
    nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    bundle_dir = os.path.dirname(os.path.dirname(nb_path))
bundle_dir = ws_path(bundle_dir)
if not os.path.exists(f"{bundle_dir}/databricks.yml"):
    raise FileNotFoundError(f"No databricks.yml in {bundle_dir}; set the bundle_dir widget to the genie-build-waves folder.")

# The bundle's own helpers: the same validation and checks that setup and deploy_instance.py use.
sys.path.insert(0, f"{bundle_dir}/src/notebooks")
import instance_guard  # noqa: E402
import monitor_grants  # noqa: E402
import waves_manifest  # noqa: E402

with open(vars_file) as f:
    bundle_vars = json.load(f)

problems = [f"missing or empty: {k}" for k in REQUIRED_VARS if not str(bundle_vars.get(k) or "").strip()]
problems += monitor_grants.monitor_var_problems(bundle_vars)
if problems:
    raise ValueError("vars file problems:\n  " + "\n  ".join(problems))

suffix = bundle_vars["instance_suffix"]
build_root = ws_path(bundle_vars["build_root"])
manifest_path = ws_path(bundle_vars["manifest_path"])

# Instance root_path: databricks.yml sets dev to ~/.bundle/<bundle><suffix>/<target>; other targets
# set their own root_path, so pass it explicitly.
root_path = dbutils.widgets.get("root_path").strip()
if not root_path:
    if target != "dev":
        raise ValueError(f"Set the root_path widget for target {target!r} (its workspace.root_path in databricks.yml).")
    root_path = f"/Workspace/Users/{me}/.bundle/{BUNDLE_NAME}{suffix}/{target}"
root_path = ws_path(root_path)

print(f"User          : {me}")
print(f"Bundle folder : {bundle_dir}")
print(f"Vars file     : {vars_file}")
print(f"Target        : {target}    action: {action}")
print(f"Instance root : {root_path}")
for k in sorted(bundle_vars):
    print(f"  {k:22s} = {bundle_vars[k]}")
if not bundle_vars.get("monitor_principal"):
    print("\nNo monitor_principal: the build job gets no monitor grant and setup grants no skill access.")

In [0]:
# The same validator the setup job runs, so errors show here before any job runs.
with open(manifest_path) as f:
    manifest = json.load(f)

errors, warnings = waves_manifest.validate_manifest(manifest)

# File checks setup does not make: every runnable session's plan file and every context file exist.
plans_dir = manifest.get("plans_dir", "docs/plans")
for wave in waves_manifest.runnable_waves(manifest):
    for s in wave.get("sessions") or []:
        plan = s.get("plan_file")
        if plan and not os.path.exists(f"{build_root}/{plans_dir}/{plan}"):
            errors.append(f"session {s.get('session_id')}: plan file not found: {plans_dir}/{plan}")
for cf in manifest.get("context_files") or []:
    if not os.path.exists(f"{build_root}/{cf}"):
        warnings.append(f"context file not found: {cf}")

# The manifest's branch is informational, but a mismatch usually means a stale manifest or vars file.
if manifest.get("branch") and manifest["branch"] != bundle_vars["build_branch"]:
    warnings.append(f"manifest branch {manifest['branch']!r} != vars build_branch {bundle_vars['build_branch']!r}")

print(waves_manifest.manifest_summary(manifest))
for m in warnings:
    print(f"  WARN  {m}")
for m in errors:
    print(f"  ERROR {m}")
if errors:
    raise ValueError(f"{len(errors)} manifest error(s): fix {manifest_path} and re-run this cell")
print(f"\nManifest OK ({len(warnings)} warning(s))")

In [0]:
# deploy_instance.py calls the Databricks CLI, which only runs in the web terminal (not on
# serverless, and not through Genie Code's CLI tool, whose state cache it cannot clear).
q = shlex.quote
base = f"python scripts/deploy_instance.py --target {q(target)} --vars {q(vars_file)}"
print("Run in the web terminal, from the genie-build-waves folder:\n")
print(f"  cd {q(bundle_dir)}")
print(f"  {base} --check-only    # 1. checks only: job IDs must belong to this instance (none on a first deploy)")
print(f"  {base}                 # 2. deploy")
print("\nThen re-run the next cell, set action = run_setup, and run the rest.")
print("Never add --force unless you have read README 'One deployment per project'.")

In [0]:
import base64

from databricks.sdk.errors import NotFound
from databricks.sdk.service.workspace import ExportFormat

# Job IDs come from this instance's remote bundle state, not from a name search: names are not
# unique across instances, and dev deploys add a "[dev <user>]" prefix.
JOB_KEYS = {"build": "build_waves", "setup": "setup_build_waves", "smoke": "build_waves_smoke"}

try:
    exported = w.workspace.export(api_path(f"{root_path}/state/resources.json"), format=ExportFormat.AUTO)
    remote_state = json.loads(base64.b64decode(exported.content))
except NotFound:
    remote_state = None

state_ids = instance_guard.state_job_ids(remote_state)
if not state_ids:
    raise RuntimeError(f"No deployed jobs in {root_path}/state. Deploy first (previous cell), then re-run this cell.")

job_ids, guard = {}, []
for role, key in JOB_KEYS.items():
    jid = state_ids.get(key)
    if not jid:
        guard.append(f"{key}: not in the instance's state")
        continue
    job = w.jobs.get(job_id=int(jid))
    job_ids[role] = int(jid)
    print(f"{role:6s} {jid:>18s}  {job.settings.name}")
    md = job.settings.deployment.metadata_file_path if job.settings.deployment else None
    if md and not instance_guard.under_root(md, root_path):
        guard.append(f"{key}: last deployed from {md}, not this instance")
    if role == "build":
        params = {p.name: p.default for p in (job.settings.parameters or [])}
        if ws_path(params.get("manifest_path", "")) != manifest_path:
            guard.append(f"build job manifest_path default {params.get('manifest_path')!r} != vars manifest_path")

if guard:
    raise RuntimeError("These jobs do not look like this instance's:\n  " + "\n  ".join(guard)
                       + "\nRun deploy_instance.py --check-only and read README 'One deployment per project'.")
host = w.config.host.rstrip("/")
print(f"\nSetup job: {host}/jobs/{job_ids['setup']}")

In [0]:
from datetime import timedelta

from databricks.sdk.errors import OperationFailed

setup_result = None
if action != "run_setup":
    print("action = check: setup not run. Set action = run_setup after the deploy to run it.")
else:
    setup_id = job_ids["setup"]
    if list(w.jobs.list_runs(job_id=setup_id, active_only=True)):
        raise RuntimeError(f"A setup run is already active: {host}/jobs/{setup_id}")

    waiter = w.jobs.run_now(job_id=setup_id, job_parameters={"run_smoke": run_smoke})
    run_id = waiter.response.run_id
    print(f"Setup run: {host}/jobs/{setup_id}/runs/{run_id}  (waiting up to {wait_minutes} min)")
    try:
        waiter.result(timeout=timedelta(minutes=wait_minutes))
    except OperationFailed:
        pass  # read the task output below for the reason
    except TimeoutError:
        raise RuntimeError(f"Setup still running after {wait_minutes} min; check the run page, then re-run this cell "
                           "with action = check and the next cell.")

    run = w.jobs.get_run(run_id=run_id)
    task = next(t for t in run.tasks if t.task_key == "setup")
    out = w.jobs.get_run_output(run_id=task.run_id)
    state = run.state.result_state.value if run.state and run.state.result_state else run.state.life_cycle_state.value
    print(f"Result: {state}")
    if out.error:
        raise RuntimeError(f"Setup failed:\n{out.error}\n{(out.error_trace or '')[-2000:]}")
    setup_result = json.loads(out.notebook_output.result)
    print("Setup succeeded.")

In [0]:
summary = {
    "project_name": bundle_vars["project_name"],
    "instance_suffix": suffix,
    "target": target,
    "root_path": root_path,
    "build_job_id": job_ids.get("build"),
    "setup_job_id": job_ids.get("setup"),
    "smoke_job_id": job_ids.get("smoke"),
    "build_job_url": f"{host}/jobs/{job_ids.get('build')}",
}
if setup_result:
    r = setup_result
    if str(r.get("build_job_id")) != str(job_ids.get("build")):
        print(f"WARN: setup patched build job {r.get('build_job_id')}, state says {job_ids.get('build')}")
    summary.update({
        "skills_published": r.get("skills_published"),
        "configuration_ids": r.get("configuration_ids"),
        "patched_tasks": r.get("patched"),
        "monitor_grants": r.get("monitor_grants"),
        "smoke_result_state": (r.get("smoke_result") or {}).get("result_state"),
    })
    mg = r.get("monitor_grants") or {}
    for g in mg.get("failed") or []:
        print(f"WARN: monitor grant failed: {g}")
    if (r.get("smoke_result") or {}).get("result_state") not in (None, "SUCCESS"):
        print(f"WARN: smoke job result {r['smoke_result'].get('result_state')}")

print(json.dumps(summary, indent=2, default=str))

## Next steps

* Re-run setup after **every** deploy: a deploy resets the build job to its placeholder task.
* External monitor (README "External monitor (optional)"): the deploy grants `monitor_principal` its `monitor_job_level` on the build job, and setup grants it read on the monitor skill (see `monitor_grants` above). Grant it `CAN_READ` on `build_root` yourself.
* Start the first build run from the Jobs UI, or let a level 4 monitor start it. Do not start one here while a monitor is driving the build.
* Put `build_job_id` and the published skill names in the project's memory file and the monitor's instruction.